## Classification of a set of Text Documents into known classes (You may use any of the Classification algorithms like Naive Bayes, Max Entropy, Rochio's, Support Vector Machine). Standard Datasets will have to be used to show the results.

# Base Work

In [11]:
# Base Work

import os
import time
import chardet
import json
import re
import tempfile
from preprocess import TextProcessor
from sklearn.feature_extraction.text import TfidfVectorizer

# ========== Document Processing ==========
def parse_newsgroup_file(file_path, encoding):
    """
    Parse a newsgroup file into individual documents/articles
    Based on the "From:" header pattern seen in your sample
    """
    with open(file_path, 'r', encoding=encoding, errors='replace') as f:
        content = f.read()
    
    # Split on "From:" lines (this is the standard newsgroup format)
    # Use positive lookahead to keep the "From:" in the next document
    documents = re.split(r'(?=From:\s+)', content)
    
    # The first element might be empty or metadata, so filter it out
    if documents and not documents[0].strip():
        documents = documents[1:]
    
    # Filter out empty or very short documents
    documents = [doc.strip() for doc in documents if len(doc.strip()) > 200]
    
    return documents

def process_newsgroup_folder(folder_path, save_folder="processed_docs", max_docs_per_category):
    """
    Process newsgroup files, splitting each into individual documents
    Uses temporary files to work with TextProcessor
    """
    all_documents = {}  # doc_id -> list of processed words
    document_metadata = {}  # doc_id -> (original_file, category)
    os.makedirs(save_folder, exist_ok=True)
    
    print("Processing newsgroup files...")
    
    # Create a temporary directory for individual document files
    with tempfile.TemporaryDirectory() as temp_dir:
        for filename in os.listdir(folder_path):
            if filename.endswith(".txt"):
                file_path = os.path.join(folder_path, filename)
                category = filename.replace('.txt', '')
                
                # Detect encoding
                with open(file_path, 'rb') as f:
                    raw_data = f.read()
                    result = chardet.detect(raw_data)
                    file_encoding = result['encoding']
                
                if file_encoding is None:
                    file_encoding = "utf-8"
                
                print(f"Processing {filename} with encoding: {file_encoding}")
                
                # Parse the file into individual documents
                raw_documents = parse_newsgroup_file(file_path, file_encoding)
                print(f"  Found {len(raw_documents)} documents in {filename}")
                
                # Process each document (limit to avoid too many files)
                processed_count = 0
                for doc_index, raw_doc in enumerate(raw_documents):
                    if processed_count >= max_docs_per_category:
                        break
                    
                    # Create a unique document ID
                    doc_id = f"{category}_doc{doc_index:04d}"
                    
                    # Extract subject from the document for better identification
                    subject_match = re.search(r'Subject:\s*(.*)', raw_doc)
                    subject = subject_match.group(1).strip() if subject_match else f"Document_{doc_index}"
                    
                    # Save as temporary file
                    temp_file_path = os.path.join(temp_dir, f"{doc_id}.txt")
                    with open(temp_file_path, 'w', encoding='utf-8') as f:
                        f.write(raw_doc)
                    
                    # Process with TextProcessor
                    try:
                        processor = TextProcessor(temp_file_path, encoding='utf-8')
                        processor.process_all()
                        processed_doc = processor.final_processed
                        
                        # Only keep documents that have enough content after processing
                        if len(processed_doc) >= 10:  # At least 10 words
                            # Store the processed document
                            all_documents[doc_id] = processed_doc
                            document_metadata[doc_id] = (filename, category, subject)
                            
                            # Save individual document
                            doc_path = os.path.join(save_folder, f"{doc_id}_processed.json")
                            with open(doc_path, 'w', encoding='utf-8') as f:
                                json.dump(processed_doc, f, ensure_ascii=False, indent=2)
                            
                            processed_count += 1
                            
                    except Exception as e:
                        print(f"  Error processing {doc_id}: {e}")
                        continue
                
                print(f"  Completed processing {processed_count} documents from {filename}")
    
    # Save metadata
    os.makedirs("indexes", exist_ok=True)
    with open("indexes/document_metadata.json", "w", encoding="utf-8") as f:
        json.dump(document_metadata, f, ensure_ascii=False, indent=2)
    
    print(f"\n✅ Processed {len(all_documents)} total documents")
    return all_documents, document_metadata

# ========== Main Execution ==========
if __name__ == "__main__":
    folder_path = r"C:/Users/DELL/OneDrive/Documents/IRS LAB/20NewsGroups"
    
    # Process newsgroup files (this will split them into individual documents)
    all_documents, document_metadata = process_newsgroup_folder(
        folder_path, 
        max_docs_per_category=100  # Process up to 1-0 documents per category
    )
    
    # Create labels from metadata
    labels = {}
    for doc_id, (filename, category, subject) in document_metadata.items():
        labels[doc_id] = category
    
    print(f"Created labels for {len(labels)} documents")
    print(f"Unique categories: {list(set(labels.values()))}")

    # ========== Global Inverted Index ==========
    print("\nBuilding global inverted index...")
    start_time = time.time()
    
    def build_inverted_index_with_frequency_and_total(docs):
        index = {}
        for doc_name, words in docs.items():
            for word in words:
                if word not in index:
                    index[word] = {"total": 0, "df": 0}
                if doc_name not in index[word]:
                    index[word][doc_name] = 0
                    index[word]["df"] += 1
                index[word][doc_name] += 1
                index[word]["total"] += 1
        return index

    global_index = build_inverted_index_with_frequency_and_total(all_documents)
    
    # Save index
    os.makedirs("indexes", exist_ok=True)
    with open("indexes/global_index.json", "w", encoding="utf-8") as f:
        json.dump(global_index, f, indent=2, ensure_ascii=False)

    end_time = time.time()
    print(f"✅ Global index built in {end_time - start_time:.2f} seconds")
    print("✅ Global index saved in indexes/global_index.json")

    # ========== TF-IDF with Scikit-learn ==========
    print("\nStarting TF-IDF vectorization with scikit-learn...")
    start_time = time.time()
    
    # Convert to sklearn format
    documents_as_strings = []
    filenames = []
    
    for doc_name, word_list in all_documents.items():
        documents_as_strings.append(" ".join(word_list))
        filenames.append(doc_name)

    # Create TF-IDF vectors
    vectorizer = TfidfVectorizer(
        max_features=10000,
        min_df=2,
        max_df=0.8,
        stop_words='english',
        norm='l2',
        use_idf=True,
        smooth_idf=True
    )

    tfidf_matrix = vectorizer.fit_transform(documents_as_strings)
    vocabulary = vectorizer.get_feature_names_out()

    # Convert to dictionary format
    tfidf_vectors = {}
    for i, doc_name in enumerate(filenames):
        tfidf_vectors[doc_name] = tfidf_matrix[i].toarray()[0].tolist()

    # Save everything
    with open("indexes/tfidf_vectors_sklearn.json", "w", encoding="utf-8") as f:
        json.dump(tfidf_vectors, f, indent=2, ensure_ascii=False)

    with open("indexes/vocabulary_sklearn.json", "w", encoding="utf-8") as f:
        json.dump(vocabulary.tolist(), f, ensure_ascii=False, indent=2)

    import pickle
    with open("indexes/tfidf_vectorizer.pkl", "wb") as f:
        pickle.dump(vectorizer, f)
    
    end_time = time.time()
    print(f"✅ TF-IDF completed in {end_time - start_time:.2f} seconds")
    print(f"Vocabulary Size: {len(vocabulary)}")
    print(f"TF-IDF vectors created: {len(tfidf_vectors)}")

    # ========== Final Summary ==========
    print("\n" + "="*50)
    print("PIPELINE COMPLETED SUCCESSFULLY!")
    print("="*50)
    print(f"Documents processed: {len(all_documents)}")
    print(f"Unique words in global index: {len(global_index)}")
    print(f"Vocabulary size (after filtering): {len(vocabulary)}")
    print("\nFiles created in 'processed_docs' folder:")
    print("- Individual processed documents (one per original file)")
    print("\nFiles created in 'indexes' folder:")
    print("- global_index.json")
    print("- tfidf_vectors_sklearn.json") 
    print("- vocabulary_sklearn.json")
    print("- tfidf_vectorizer.pkl")
    print("- all_documents_mapping.json")
    print("="*50)

Processing newsgroup files...
Processing alt.atheism.txt with encoding: utf-8
  Found 2405 documents in alt.atheism.txt
  Completed processing 100 documents from alt.atheism.txt
Processing comp.graphics.txt with encoding: ISO-8859-1
  Found 2479 documents in comp.graphics.txt
  Completed processing 100 documents from comp.graphics.txt
Processing comp.os.ms-windows.misc.txt with encoding: ascii
  Found 1978 documents in comp.os.ms-windows.misc.txt
  Completed processing 100 documents from comp.os.ms-windows.misc.txt
Processing comp.sys.ibm.pc.hardware.txt with encoding: ISO-8859-1
  Found 1972 documents in comp.sys.ibm.pc.hardware.txt
  Completed processing 100 documents from comp.sys.ibm.pc.hardware.txt
Processing comp.sys.mac.hardware.txt with encoding: Windows-1252
  Found 1918 documents in comp.sys.mac.hardware.txt
  Completed processing 100 documents from comp.sys.mac.hardware.txt
Processing comp.windows.x.txt with encoding: Windows-1252
  Found 1954 documents in comp.windows.x.txt

# Manual NB Classifier

In [17]:
import json
import numpy as np
import pandas as pd # Import pandas
from collections import Counter

# Import the ManualNaiveBayes class and utility functions from the new file
from manual_naive_bayes_utils import ManualNaiveBayes, train_test_split

# ========== Load Processed Data ==========
# print("Loading processed data...")

# Load the document mapping
with open("indexes/document_metadata.json", "r", encoding="utf-8") as f:
    document_metadata = json.load(f)

# Load processed documents
all_documents = {}
for doc_id, (filename, category, subject) in document_metadata.items():
    with open(f"processed_docs/{doc_id}_processed.json", "r", encoding="utf-8") as f:
        all_documents[doc_id] = json.load(f)


# ========== Prepare Labels ==========
# Extract class labels from metadata
labels = {}
for doc_id, (filename, category, subject) in document_metadata.items():
    labels[doc_id] = category

# Load vocabulary (limited to top 10,000 features from sklearn)
with open("indexes/vocabulary_sklearn.json", "r", encoding="utf-8") as f:
    vocabulary = json.load(f)

# print(f"Loaded {len(all_documents)} documents with vocabulary size: {len(vocabulary)}")

# Get all unique classes
classes = list(set(labels.values()))
# print(f"Found {len(classes)} classes: {classes}")

# ========== Main Execution ==========
if __name__ == "__main__":
    # Split data into training and testing
    train_docs, test_docs, train_labels, test_labels = train_test_split(
        all_documents, labels, test_size=0.2, random_seed=42
    )

    # print(f"Training set: {len(train_docs)} documents")
    # print(f"Test set: {len(test_docs)} documents")

    # Train Naive Bayes classifier
    nb_classifier = ManualNaiveBayes(vocabulary)
    nb_classifier.train(train_docs, train_labels)

    # Save the trained model
    # nb_classifier.save_model("naive_bayes_model.json")
    # print("✅ Model saved as naive_bayes_model.json")

    # Evaluate on test data
    results = nb_classifier.evaluate(test_docs, test_labels)

    print(f"\n=== EVALUATION RESULTS ===") # Modified print statement

    # Create a DataFrame for overall metrics
    overall_metrics = {
        'Model': ['Naive Bayes'],
        'Accuracy': [results['accuracy']],
        'Precision': [np.mean([m['precision'] for m in results['class_metrics'].values()])], # Calculate average precision
        'Recall': [np.mean([m['recall'] for m in results['class_metrics'].values()])],     # Calculate average recall
        'F1-Score': [np.mean([m['f1'] for m in results['class_metrics'].values()])]         # Calculate average F1-score
    }
    overall_metrics_df = pd.DataFrame(overall_metrics)
    display(overall_metrics_df.round(4)) # Display the DataFrame

    # print(f"Accuracy: {results['accuracy']:.4f} ({int(results['accuracy'] * len(test_docs))}/{len(test_docs)})") # Commented out

    # print(f"\n=== PER-CLASS METRICS ===") # Commented out
    # for class_label, metrics in results['class_metrics'].items(): # Commented out
    #     print(f"Class: {class_label}") # Commented out
    #     print(f"  Precision: {metrics['precision']:.4f}") # Commented out
    #     print(f"  Recall:    {metrics['recall']:.4f}") # Commented out
    #     print(f"  F1-score:  {metrics['f1']:.4f}") # Commented out
    #     print(f"  Support:   {metrics['support']}") # Commented out
    #     print() # Commented out

    # print("=== SAMPLE PREDICTIONS ===") # Commented out
    # for i, (filename, true_label, predicted_label) in enumerate(results['predictions'][:5]): # Commented out
    #     status = "✓" if true_label == predicted_label else "✗" # Commented out
    #     print(f"{status} {filename}: True={true_label}, Predicted={predicted_label}") # Commented out

Training Naive Bayes classifier...
Training completed!

=== EVALUATION RESULTS ===


,Model,Accuracy,Precision,Recall,F1-Score
0,Naive Bayes,0.8825,0.8874,0.8856,0.879


In [1]:
import json
import numpy as np
from collections import Counter

# Import the ManualNaiveBayes class and utility functions from the new file
from manual_naive_bayes_utils import ManualNaiveBayes, train_test_split

# ========== Load Processed Data ==========
print("Loading processed data...")

# Load the document mapping
with open("indexes/document_metadata.json", "r", encoding="utf-8") as f:
    document_metadata = json.load(f)

# Load processed documents
all_documents = {}
for doc_id, (filename, category, subject) in document_metadata.items():
    with open(f"processed_docs/{doc_id}_processed.json", "r", encoding="utf-8") as f:
        all_documents[doc_id] = json.load(f)


# ========== Prepare Labels ==========
# Extract class labels from metadata
labels = {}
for doc_id, (filename, category, subject) in document_metadata.items():
    labels[doc_id] = category

# Load vocabulary (limited to top 10,000 features from sklearn)
with open("indexes/vocabulary_sklearn.json", "r", encoding="utf-8") as f:
    vocabulary = json.load(f)

print(f"Loaded {len(all_documents)} documents with vocabulary size: {len(vocabulary)}")

# Get all unique classes
classes = list(set(labels.values()))
print(f"Found {len(classes)} classes: {classes}")

# ========== Main Execution ==========
if __name__ == "__main__":
    # Split data into training and testing
    train_docs, test_docs, train_labels, test_labels = train_test_split(
        all_documents, labels, test_size=0.2, random_seed=42
    )

    print(f"Training set: {len(train_docs)} documents")
    print(f"Test set: {len(test_docs)} documents")

    # Train Naive Bayes classifier
    nb_classifier = ManualNaiveBayes(vocabulary)
    nb_classifier.train(train_docs, train_labels)

    # Save the trained model
    nb_classifier.save_model("naive_bayes_model.json")
    print("✅ Model saved as naive_bayes_model.json")

    # Evaluate on test data
    results = nb_classifier.evaluate(test_docs, test_labels)

    print(f"\n=== EVALUATION RESULTS ===")
    print(f"Accuracy: {results['accuracy']:.4f} ({int(results['accuracy'] * len(test_docs))}/{len(test_docs)})")

    print(f"\n=== PER-CLASS METRICS ===")
    for class_label, metrics in results['class_metrics'].items():
        print(f"Class: {class_label}")
        print(f"  Precision: {metrics['precision']:.4f}")
        print(f"  Recall:    {metrics['recall']:.4f}")
        print(f"  F1-score:  {metrics['f1']:.4f}")
        print(f"  Support:   {metrics['support']}")
        print()

    print("=== SAMPLE PREDICTIONS ===")
    for i, (filename, true_label, predicted_label) in enumerate(results['predictions'][:5]):
        status = "✓" if true_label == predicted_label else "✗"
        print(f"{status} {filename}: True={true_label}, Predicted={predicted_label}")

Loading processed data...
Loaded 2000 documents with vocabulary size: 10000
Found 20 classes: ['sci.space', 'talk.politics.guns', 'rec.motorcycles', 'comp.graphics', 'soc.religion.christian', 'sci.med', 'sci.electronics', 'sci.crypt', 'talk.religion.misc', 'rec.sport.hockey', 'comp.os.ms-windows.misc', 'talk.politics.misc', 'rec.autos', 'comp.sys.mac.hardware', 'comp.windows.x', 'alt.atheism', 'rec.sport.baseball', 'talk.politics.mideast', 'comp.sys.ibm.pc.hardware', 'misc.forsale']
Training set: 1600 documents
Test set: 400 documents
Training Naive Bayes classifier...
Training completed!
✅ Model saved as naive_bayes_model.json

=== EVALUATION RESULTS ===
Accuracy: 0.8825 (353/400)

=== PER-CLASS METRICS ===
Class: talk.politics.guns
  Precision: 0.9500
  Recall:    0.9500
  F1-score:  0.9500
  Support:   20

Class: sci.space
  Precision: 0.9167
  Recall:    0.8462
  F1-score:  0.8800
  Support:   26

Class: rec.motorcycles
  Precision: 0.9130
  Recall:    1.0000
  F1-score:  0.9545
  

# Manual Rocchio Classifier:

In [11]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Import the RocchioClassifier from the new file
from rocchio_classifier_utils import RocchioClassifier

# Load the 20 Newsgroups dataset
categories = ['sci.space', 'comp.graphics', 'rec.sport.baseball', 'talk.politics.guns']
newsgroups = fetch_20newsgroups(subset='all', categories=categories, shuffle=True, random_state=42)
X, y = newsgroups.data, newsgroups.target

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Vectorize the text data
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Get feature names and convert to dense arrays for manual NB
feature_names = vectorizer.get_feature_names_out()
X_train_dense = X_train_vec.toarray()
X_test_dense = X_test_vec.toarray()

# Train and evaluate Rocchio classifier
print("Training Rocchio Classifier...")
rocchio = RocchioClassifier()
rocchio.fit(X_train_dense, y_train)
y_pred_rocchio = rocchio.predict(X_test_dense)

# Calculate metrics for Rocchio
def calculate_metrics(y_true, y_pred, model_name):
    return {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'Recall': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'F1-Score': f1_score(y_true, y_pred, average='weighted', zero_division=0)
    }

rocchio_results = calculate_metrics(y_test, y_pred_rocchio, 'Rocchio')

print("\nRocchio Classifier Performance:")
print(pd.DataFrame([rocchio_results]).round(4))

Training Rocchio Classifier...

Rocchio Classifier Performance:
     Model  Accuracy  Precision  Recall  F1-Score
0  Rocchio    0.9466     0.9484  0.9466    0.9465


In [19]:
import numpy as np
import pandas as pd
from collections import defaultdict
import re
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')


# Load the 20 Newsgroups dataset
categories = ['sci.space', 'comp.graphics', 'rec.sport.baseball', 'talk.politics.guns']
newsgroups = fetch_20newsgroups(subset='all', categories=categories, shuffle=True, random_state=42)
X, y = newsgroups.data, newsgroups.target

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Vectorize the text data
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Get feature names and convert to dense arrays for manual NB
feature_names = vectorizer.get_feature_names_out()
X_train_dense = X_train_vec.toarray()
X_test_dense = X_test_vec.toarray()


# Train and evaluate Logistic Regression (Max Entropy)
print("Training Logistic Regression...")
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_vec, y_train)
y_pred_lr = lr.predict(X_test_vec)

# Train and evaluate SVM
print("Training SVM...")
svm = SVC(kernel='linear', random_state=42)
svm.fit(X_train_vec, y_train)
y_pred_svm = svm.predict(X_test_vec)


# Calculate metrics for each classifier
def calculate_metrics(y_true, y_pred, model_name):
    return {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'Recall': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'F1-Score': f1_score(y_true, y_pred, average='weighted', zero_division=0)
    }

# Create results dataframe
results = []
results.append(calculate_metrics(y_test, y_pred_lr, 'Logistic Regression'))
results.append(calculate_metrics(y_test, y_pred_svm, 'SVM'))

results_df = pd.DataFrame(results)
print("\nComparison of Classification Algorithms:")
print(results_df.round(4))

Training Logistic Regression...
Training SVM...

Comparison of Classification Algorithms:
                 Model  Accuracy  Precision  Recall  F1-Score
0  Logistic Regression    0.9629     0.9633  0.9629    0.9628
1                  SVM    0.9741     0.9745  0.9741    0.9741
